# 🔧 File 02 — Tiền xử lý dữ liệu (Preprocessing)
**Brain Tumor Classification MRI Dataset**

Mục tiêu:
- Resize ảnh về kích thước đồng nhất
- Normalize pixel về [0, 1]
- Tách Validation set từ Training set
- Data Augmentation (tăng cường dữ liệu)
- Tạo data pipeline sẵn sàng cho model

## 📦 Phần A — Import thư viện

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings('ignore')

print(f'✅ TensorFlow version: {tf.__version__}')

## ⚙️ Phần B — Cấu hình tham số

Tất cả các thông số quan trọng đặt ở đây để dễ thay đổi sau.

In [ ]:
# ============================================================
# CONFIG - use project-local paths only
# ============================================================
PROJECT_ROOT = Path.cwd()
DATA_DIR  = PROJECT_ROOT / 'data'
TRAIN_DIR = DATA_DIR / 'Training'
TEST_DIR  = DATA_DIR / 'Testing'
CSV_DIR   = PROJECT_ROOT
PROCESSED_DIR = PROJECT_ROOT / 'data_processed_roi'

IMG_SIZE    = (240, 240)
BATCH_SIZE  = 32
VAL_SPLIT   = 0.2
RANDOM_SEED = 42
APPLY_ROI_CROP = True
FORCE_REPROCESS = False
APPLY_INTENSITY_NORMALIZATION = True
APPLY_CLAHE = False  # Set True only after checking the preview; CLAHE can amplify noise.
PERCENTILE_RANGE = (1, 99)
BLACK_PIXEL_THRESHOLD = 8

CLASSES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
CLASS_LABELS = {
    'glioma_tumor':     'Glioma',
    'meningioma_tumor': 'Meningioma',
    'no_tumor':         'Khong co u',
    'pituitary_tumor':  'Pituitary'
}

assert TRAIN_DIR.exists(), f'Missing training folder: {TRAIN_DIR}'
assert TEST_DIR.exists(),  f'Missing testing folder: {TEST_DIR}'

print('Config:')
print(f'   Project root : {PROJECT_ROOT}')
print(f'   Data dir     : {DATA_DIR}')
print(f'   Processed dir: {PROCESSED_DIR}')
print(f'   Image size   : {IMG_SIZE}')
print(f'   Batch size   : {BATCH_SIZE}')
print(f'   Validation   : {int(VAL_SPLIT * 100)}% of Training')
print(f'   Num classes  : {len(CLASSES)}')
print(f'   ROI crop     : {APPLY_ROI_CROP}')
print(f'   Intensity norm: {APPLY_INTENSITY_NORMALIZATION}')
print(f'   CLAHE        : {APPLY_CLAHE}')

## 📋 Phần C — Tạo danh sách file ảnh

Thu thập đường dẫn tất cả ảnh và nhãn tương ứng vào DataFrame.

In [ ]:
# ============================================================
# Build DataFrame with original image paths and labels
# ============================================================
def build_dataframe(base_dir, classes):
    """
    Scan folders and return a DataFrame:
    - filepath: image path
    - label   : class name (glioma_tumor, ...)
    """
    records = []
    for cls in classes:
        cls_path = base_dir / cls
        for ext in ['*.jpg', '*.jpeg', '*.png']:
            for img_path in cls_path.glob(ext):
                records.append({
                    'filepath': str(img_path.resolve()),
                    'label':    cls
                })
    df = pd.DataFrame(records)
    df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    return df

# Keep raw DataFrames for before/after preprocessing checks.
df_train_all_raw = build_dataframe(TRAIN_DIR, CLASSES)
df_test_raw      = build_dataframe(TEST_DIR,  CLASSES)

print(f'Raw Training + Val: {len(df_train_all_raw)} images')
print(f'Raw Testing       : {len(df_test_raw)} images')
print()
print('Raw data sample:')
print(df_train_all_raw.head())


## Phan D - ROI crop va histogram pixel sau tien xu ly

Anh MRI co nhieu nen den va vien ngoai sang. Neu dua thang vao model, model co the hoc nen/vien thay vi vung nao hoac khoi u.

Phan nay crop vung foreground lon nhat, resize ve cung kich thuoc, roi so sanh histogram pixel truoc/sau crop.


In [ ]:
# ============================================================
# ROI crop + robust intensity normalization
# - ROI crop reduces black background and scanner framing bias.
# - Percentile normalization reduces brightness/contrast differences across scans.
# - Optional CLAHE can enhance local contrast but may amplify noise, so it is off by default.
# ============================================================
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def crop_foreground_roi(img_bgr, margin=0.08, min_area_ratio=0.03):
    """Crop the largest foreground region; return original image if crop is unreliable."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    mask = (blur > 5).astype(np.uint8) * 255
    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img_bgr

    contour = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(contour)
    h, w = gray.shape[:2]
    if area < min_area_ratio * h * w:
        return img_bgr

    x, y, bw, bh = cv2.boundingRect(contour)
    pad = int(max(bw, bh) * margin)
    x1 = max(x - pad, 0)
    y1 = max(y - pad, 0)
    x2 = min(x + bw + pad, w)
    y2 = min(y + bh + pad, h)
    return img_bgr[y1:y2, x1:x2]


def robust_intensity_normalize(img_bgr, percentile_range=PERCENTILE_RANGE):
    """Clip extreme intensities and stretch useful foreground contrast to 0-255."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    foreground = gray[gray > BLACK_PIXEL_THRESHOLD]
    if foreground.size < 50:
        foreground = gray.reshape(-1)

    lo, hi = np.percentile(foreground, percentile_range)
    if hi <= lo:
        return img_bgr

    img = img_bgr.astype(np.float32)
    img = (img - lo) / (hi - lo) * 255.0
    return np.clip(img, 0, 255).astype(np.uint8)


def apply_clahe_luminance(img_bgr, clip_limit=1.5, tile_grid_size=(8, 8)):
    """Apply conservative CLAHE to luminance only."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_channel = clahe.apply(l_channel)
    lab = cv2.merge([l_channel, a_channel, b_channel])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def preprocess_image(img_bgr):
    img = crop_foreground_roi(img_bgr) if APPLY_ROI_CROP else img_bgr
    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
    if APPLY_INTENSITY_NORMALIZATION:
        img = robust_intensity_normalize(img)
    if APPLY_CLAHE:
        img = apply_clahe_luminance(img)
    return img


def preprocess_dataframe(df_raw, split_name):
    records = []
    out_split_dir = PROCESSED_DIR / split_name
    out_split_dir.mkdir(parents=True, exist_ok=True)

    for idx, row in df_raw.reset_index(drop=True).iterrows():
        src_path = Path(row['filepath'])
        label = row['label']
        out_dir = out_split_dir / label
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f'{idx:05d}_{src_path.stem}.png'

        if FORCE_REPROCESS or not out_path.exists():
            img = cv2.imread(str(src_path))
            if img is None:
                print(f'WARNING: Cannot read image: {src_path}')
                continue
            processed = preprocess_image(img)
            cv2.imwrite(str(out_path), processed)

        records.append({
            'filepath': str(out_path.resolve()),
            'label': label,
            'source_filepath': str(src_path.resolve())
        })

        if (idx + 1) % 500 == 0:
            print(f'   {split_name}: processed {idx + 1}/{len(df_raw)} images')

    return pd.DataFrame(records)


if APPLY_ROI_CROP or APPLY_INTENSITY_NORMALIZATION or APPLY_CLAHE:
    print('Creating/using processed dataset...')
    df_train_all = preprocess_dataframe(df_train_all_raw, 'Training')
    df_test      = preprocess_dataframe(df_test_raw,      'Testing')
else:
    print('All preprocessing disabled; using raw image paths.')
    df_train_all = df_train_all_raw.copy()
    df_test      = df_test_raw.copy()

print(f'Training + Val after preprocessing: {len(df_train_all)} images')
print(f'Testing after preprocessing       : {len(df_test)} images')
print(df_train_all.head())


## Phan E - So sanh truoc/sau crop va histogram pixel

Histogram sau crop co y nghia hon histogram raw vi nen den da giam bot. No van khong chi vi tri u, nhung giup kiem tra anh con bi bias sang/toi hoac nen den qua nhieu khong.


In [ ]:
# ============================================================
# Visual QA: original vs processed image
# ============================================================
fig, axes = plt.subplots(len(CLASSES), 2, figsize=(8, 12))
fig.suptitle('Original image vs processed image', fontsize=13, fontweight='bold')

for row, cls in enumerate(CLASSES):
    raw_path = df_train_all_raw[df_train_all_raw['label'] == cls].iloc[0]['filepath']
    processed_path = df_train_all[df_train_all['label'] == cls].iloc[0]['filepath']

    raw_img = cv2.cvtColor(cv2.imread(raw_path), cv2.COLOR_BGR2RGB)
    processed_img = cv2.cvtColor(cv2.imread(processed_path), cv2.COLOR_BGR2RGB)

    axes[row, 0].imshow(raw_img)
    axes[row, 0].set_title(f'{CLASS_LABELS[cls]} - raw')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(processed_img)
    axes[row, 1].set_title('Processed')
    axes[row, 1].axis('off')

plt.tight_layout()
plt.savefig('preprocessing_roi_crop_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: preprocessing_roi_crop_preview.png')


# ============================================================
# Per-class processed grid: catch crop failures class by class
# ============================================================
N_GRID = 4
fig, axes = plt.subplots(len(CLASSES), N_GRID, figsize=(12, 10))
fig.suptitle('Processed samples by class', fontsize=13, fontweight='bold')

for row, cls in enumerate(CLASSES):
    subset = df_train_all[df_train_all['label'] == cls].head(N_GRID)
    for col, (_, item) in enumerate(subset.iterrows()):
        img = cv2.cvtColor(cv2.imread(item['filepath']), cv2.COLOR_BGR2RGB)
        axes[row, col].imshow(img)
        if col == 0:
            axes[row, col].set_ylabel(CLASS_LABELS[cls], fontsize=10, fontweight='bold')
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('preprocessing_processed_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: preprocessing_processed_grid.png')


# ============================================================
# Histogram before/after preprocessing
# ============================================================
N_HIST_SAMPLES = 60
fig, axes = plt.subplots(2, len(CLASSES), figsize=(16, 7), sharex=True)
fig.suptitle('Pixel intensity before and after preprocessing', fontsize=13, fontweight='bold')

quality_rows = []

for col, cls in enumerate(CLASSES):
    raw_pixels = []
    processed_pixels = []
    raw_black_ratios = []
    processed_black_ratios = []

    raw_subset = df_train_all_raw[df_train_all_raw['label'] == cls].head(N_HIST_SAMPLES)
    processed_subset = df_train_all[df_train_all['label'] == cls].head(N_HIST_SAMPLES)

    for p in raw_subset['filepath']:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, IMG_SIZE)
            raw_pixels.extend(img.flatten())
            raw_black_ratios.append((img < BLACK_PIXEL_THRESHOLD).mean())

    for p in processed_subset['filepath']:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            processed_pixels.extend(img.flatten())
            processed_black_ratios.append((img < BLACK_PIXEL_THRESHOLD).mean())

    quality_rows.append({
        'class': CLASS_LABELS[cls],
        'raw_mean': float(np.mean(raw_pixels)),
        'processed_mean': float(np.mean(processed_pixels)),
        'raw_black_ratio': float(np.mean(raw_black_ratios)),
        'processed_black_ratio': float(np.mean(processed_black_ratios))
    })

    axes[0, col].hist(raw_pixels, bins=50, color='#78909C', alpha=0.9)
    axes[0, col].axvline(np.mean(raw_pixels), color='black', linestyle='--', linewidth=1)
    axes[0, col].set_title(f'{CLASS_LABELS[cls]}\nRaw mean={np.mean(raw_pixels):.0f}', fontsize=10)

    axes[1, col].hist(processed_pixels, bins=50, color='#26A69A', alpha=0.9)
    axes[1, col].axvline(np.mean(processed_pixels), color='black', linestyle='--', linewidth=1)
    axes[1, col].set_title(f'Processed mean={np.mean(processed_pixels):.0f}', fontsize=10)
    axes[1, col].set_xlabel('Pixel value (0-255)')

axes[0, 0].set_ylabel('Raw')
axes[1, 0].set_ylabel('Processed')
plt.tight_layout()
plt.savefig('preprocessing_pixel_hist_before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: preprocessing_pixel_hist_before_after.png')

quality_df = pd.DataFrame(quality_rows)
print('Preprocessing quality summary:')
print(quality_df.to_string(index=False, formatters={
    'raw_mean': '{:.1f}'.format,
    'processed_mean': '{:.1f}'.format,
    'raw_black_ratio': '{:.1%}'.format,
    'processed_black_ratio': '{:.1%}'.format,
}))

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(quality_df))
width = 0.35
ax.bar(x - width/2, quality_df['raw_black_ratio'] * 100, width, label='Raw', color='#78909C')
ax.bar(x + width/2, quality_df['processed_black_ratio'] * 100, width, label='Processed', color='#26A69A')
ax.set_xticks(x)
ax.set_xticklabels(quality_df['class'])
ax.set_ylabel('Black pixels (%)')
ax.set_title('Black-background ratio before/after preprocessing')
ax.legend()
plt.tight_layout()
plt.savefig('preprocessing_quality_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: preprocessing_quality_stats.png')


## Phan F - Tach Validation Set

**Tai sao can Validation?**
- Training set: model hoc truc tiep
- Validation set: danh gia model sau moi epoch de dieu chinh
- Test set: danh gia cuoi cung, chi dung 1 lan

Dung `stratify=label` de dam bao ty le class bang nhau o ca 2 tap.


In [ ]:
# ============================================================
# Tách Train / Validation (stratified — giữ tỷ lệ class)
# ============================================================
df_train, df_val = train_test_split(
    df_train_all,
    test_size=VAL_SPLIT,
    random_state=RANDOM_SEED,
    stratify=df_train_all['label']   # Đảm bảo mỗi class có tỷ lệ bằng nhau
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)

print(f'✅ Training   : {len(df_train)} ảnh')
print(f'✅ Validation : {len(df_val)} ảnh')
print(f'✅ Testing    : {len(df_test)} ảnh')
print()

# Kiểm tra tỷ lệ class sau khi tách
print('Phân phối class trong Validation:')
print(df_val['label'].value_counts())

## Phan G - Data Augmentation

**Data Augmentation** tao them anh bien the tu anh goc de giam overfitting.

Chi ap dung augmentation cho Training set, khong ap dung cho Val/Test.


In [ ]:
# ============================================================
# ImageDataGenerator
# Keep augmentation conservative for MRI. Very strong transforms can
# destroy tumor location/shape cues and make Glioma/Meningioma worse.
# ============================================================
from sklearn.utils import compute_class_weight

train_datagen = ImageDataGenerator(
    rotation_range=7,
    width_shift_range=0.03,
    height_shift_range=0.03,
    shear_range=0.0,
    zoom_range=0.07,
    horizontal_flip=True,
    vertical_flip=False,
    brightness_range=[0.95, 1.05],
    fill_mode='constant',
    cval=0
)

# Val/Test are not augmented.
val_test_datagen = ImageDataGenerator()

print('ImageDataGenerator ready: conservative train augmentation, no val/test augmentation')

In [ ]:
# ============================================================
# Create data generators from DataFrames
# ============================================================
train_generator = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES,
    shuffle=True, seed=RANDOM_SEED
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_val,
    x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_test,
    x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES,
    shuffle=False
)

assert val_generator.class_indices == train_generator.class_indices
assert test_generator.class_indices == train_generator.class_indices

CLASS_INDICES  = train_generator.class_indices
INDEX_TO_CLASS = {v: k for k, v in CLASS_INDICES.items()}
print(f'Class indices: {CLASS_INDICES}')
print(f'Samples: train={train_generator.samples}, val={val_generator.samples}, test={test_generator.samples}')

labels = train_generator.classes
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(class_weights))

print('\nClass weights:')
for idx, cls in enumerate(CLASSES):
    print(f'   {CLASS_LABELS[cls]:<15}: {class_weight_dict[idx]:.3f}')

## Phan H - Kiem tra ket qua augmentation

Xem anh goc va cac bien the augment de dam bao khong bi bien dang qua muc.


In [ ]:
# ============================================================
# Hiển thị ảnh gốc vs ảnh sau augmentation
# ============================================================
sample_path = df_train.iloc[0]['filepath']
sample_img  = cv2.imread(sample_path)
sample_img  = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
sample_img  = cv2.resize(sample_img, IMG_SIZE)

# Tạo augmented versions
sample_arr = sample_img.reshape((1,) + sample_img.shape).astype('float32')
aug_gen = train_datagen.flow(sample_arr, batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Ảnh gốc (trái) và các biến thể Augmentation',
             fontsize=13, fontweight='bold')

# Hàng 1: ảnh gốc + augment #1-4
axes[0, 0].imshow(sample_img.astype('uint8'))
axes[0, 0].set_title('Ảnh gốc', fontweight='bold')
axes[0, 0].axis('off')

for j in range(1, 5):
    aug_img = next(aug_gen)[0]
    axes[0, j].imshow(np.clip(aug_img, 0, 255).astype('uint8'))  # ← sửa
    axes[0, j].set_title(f'Augment #{j}')
    axes[0, j].axis('off')

# Hàng 2: augment #5-9
for j in range(5):
    aug_img = next(aug_gen)[0]
    axes[1, j].imshow(np.clip(aug_img, 0, 255).astype('uint8'))  # ← sửa
    axes[1, j].set_title(f'Augment #{j+5}')
    axes[1, j].axis('off')

plt.tight_layout()
plt.savefig('preprocessing_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()

## Phan I - Label/path consistency-check

Kiem tra nhan trong DataFrame co khop voi thu muc chua anh processed khong. Buoc nay khong xac nhan chan doan y khoa, nhung bat loi pipeline/CSV/class mapping.


In [ ]:
# ============================================================
# Label/path consistency-check
# This verifies technical label consistency, not medical correctness.
# ============================================================
def check_label_path_consistency(df, split_name):
    rows = []
    for _, item in df.iterrows():
        path_label = Path(item['filepath']).parent.name
        rows.append({
            'split': split_name,
            'filepath': item['filepath'],
            'label': item['label'],
            'path_label': path_label,
            'ok': item['label'] == path_label
        })
    result = pd.DataFrame(rows)
    n_bad = int((~result['ok']).sum())
    print(f'{split_name}: {len(result)} rows, path/label mismatches = {n_bad}')
    return result

label_checks = pd.concat([
    check_label_path_consistency(df_train, 'train'),
    check_label_path_consistency(df_val, 'val'),
    check_label_path_consistency(df_test, 'test'),
], ignore_index=True)

label_checks.to_csv('preprocessing_label_path_check.csv', index=False)
print('Saved: preprocessing_label_path_check.csv')

bad_rows = label_checks[~label_checks['ok']]
if len(bad_rows) > 0:
    print('WARNING: Found label/path mismatches. First rows:')
    print(bad_rows.head().to_string(index=False))
else:
    print('All labels match their processed-image folders.')


## Phan I - Batch sanity-check

Xem nhanh mot batch sau preprocessing + augmentation de chac model dang nhan anh hop ly, khong bi mat vung u hoac crop qua manh.


In [ ]:
# ============================================================
# Batch sanity-check: images actually fed to the model
# Note: this checks preprocessing/augmentation quality, not medical truth.
# ============================================================
train_generator.reset()
batch_x, batch_y = next(train_generator)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
fig.suptitle('Sample batch after preprocessing + augmentation', fontsize=13, fontweight='bold')
axes = axes.flatten()

batch_rows = []
for i, ax in enumerate(axes):
    if i >= len(batch_x):
        ax.axis('off')
        continue
    img = np.clip(batch_x[i], 0, 255).astype('uint8')
    label_idx = int(np.argmax(batch_y[i]))
    label_key = CLASSES[label_idx]
    label = CLASS_LABELS[label_key]
    ax.imshow(img)
    ax.set_title(f'{label}\nclass_id={label_idx}', fontsize=9)
    ax.axis('off')
    batch_rows.append({'batch_index': i, 'class_id': label_idx, 'label': label_key})

plt.tight_layout()
plt.savefig('preprocessing_batch_sanity_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: preprocessing_batch_sanity_check.png')
print('Batch labels shown above are decoded from generator one-hot labels.')

pd.DataFrame(batch_rows).to_csv('preprocessing_batch_sanity_labels.csv', index=False)
print('Saved: preprocessing_batch_sanity_labels.csv')


## Phan K - Deterministic per-class label QA grid

Hien thi anh processed theo tung class kem ten file. Grid nay de soi nhan theo duong dan ro hon batch augmentation ngau nhien.


In [ ]:
# ============================================================
# Deterministic per-class label QA grid
# ============================================================
N_PER_CLASS = 4
fig, axes = plt.subplots(len(CLASSES), N_PER_CLASS, figsize=(14, 10))
fig.suptitle('Label QA grid: processed samples by class', fontsize=13, fontweight='bold')
qa_rows = []

for row, cls in enumerate(CLASSES):
    subset = df_train[df_train['label'] == cls].head(N_PER_CLASS)
    for col in range(N_PER_CLASS):
        ax = axes[row, col]
        if col >= len(subset):
            ax.axis('off')
            continue
        item = subset.iloc[col]
        img = cv2.cvtColor(cv2.imread(item['filepath']), cv2.COLOR_BGR2RGB)
        filename = Path(item['filepath']).name
        source_name = Path(item.get('source_filepath', item['filepath'])).name
        ax.imshow(img)
        ax.set_title(f'{CLASS_LABELS[cls]}\n{filename[:22]}', fontsize=8)
        ax.axis('off')
        qa_rows.append({
            'label': cls,
            'processed_filepath': item['filepath'],
            'source_filepath': item.get('source_filepath', ''),
            'processed_filename': filename,
            'source_filename': source_name,
        })

plt.tight_layout()
plt.savefig('preprocessing_label_qa_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: preprocessing_label_qa_grid.png')

pd.DataFrame(qa_rows).to_csv('preprocessing_label_qa_grid_samples.csv', index=False)
print('Saved: preprocessing_label_qa_grid_samples.csv')


## Phan J - Luu thong tin de dung o cac file sau


In [ ]:
# ============================================================
# Save DataFrames for 03_Model and 04_Evaluation
# ============================================================
train_csv = CSV_DIR / 'df_train.csv'
val_csv   = CSV_DIR / 'df_val.csv'
test_csv  = CSV_DIR / 'df_test.csv'

df_train.to_csv(train_csv, index=False)
df_val.to_csv(val_csv, index=False)
df_test.to_csv(test_csv, index=False)


# Save class mapping for model/evaluation notebooks
class_mapping = {
    'classes_order': CLASSES,
    'class_to_index': CLASS_INDICES,
    'index_to_class': {str(k): v for k, v in INDEX_TO_CLASS.items()},
    'index_to_display_label': {str(i): CLASS_LABELS[cls] for i, cls in enumerate(CLASSES)},
}

with open(CSV_DIR / 'class_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(class_mapping, f, indent=2, ensure_ascii=False)

pd.DataFrame([
    {
        'class_index': i,
        'class_key': cls,
        'display_label': CLASS_LABELS[cls],
    }
    for i, cls in enumerate(CLASSES)
]).to_csv(CSV_DIR / 'class_mapping.csv', index=False, encoding='utf-8')

print('  ', CSV_DIR / 'class_mapping.json')
print('  ', CSV_DIR / 'class_mapping.csv')

print('Saved CSV files:')
print('  ', train_csv, '-', len(df_train), 'images')
print('  ', val_csv,   '-', len(df_val), 'images')
print('  ', test_csv,  '-', len(df_test), 'images')
print('Processed image dir:', PROCESSED_DIR if (APPLY_ROI_CROP or APPLY_INTENSITY_NORMALIZATION or APPLY_CLAHE) else 'disabled')
print('\nPreprocessing complete. Run 03_Model_v3.ipynb next.')